# 04｜Trinity Study：4% 提領法則的學術驗證
**理論來源：Cooley, Hubbard & Walz（1998），Trinity University**

> **核心問題：退休後每年花多少比例的錢，才能讓資產撐過 30 年？**
>
> 1998 年三位教授用 1926–1995 年美股資料回測各種提領率的「存活率」。
> 結論：**4% 是 30 年期間存活率接近 100% 的最大安全提領率。**
>
> 本 Notebook 用 Shiller 145 年資料重現這個分析，
> 並加入一個往往被忽略的關鍵：**報酬序列風險（Sequence of Returns Risk）**。

## 想像你 55 歲退休，戶頭有 2,000 萬

你不用再工作了。但 2,000 萬放著不動，通膨每年會吃掉一點。
所以你決定每年提領一些錢來生活。

**問題是：每年可以提多少，才不會在你 80 歲還在世的時候把錢花光？**

太多：20 年後錢就沒了，你 75 歲開始挨餓。
太少：你 90 歲離開的時候戶頭還有 5,000 萬，那存這麼多到底為了什麼？

這不是理財廣告，而是一個真實的數學問題。

1994 年，美國財務規劃師 Bengen 用 70 年的股債歷史數據算出了一個關鍵答案。
四年後，三位德州大學教授（Cooley、Hubbard、Walz）用更嚴謹的統計全面驗證，
結論被稱為「**4% 法則**」——個人理財史上最常被引用的規則之一。

我們把它拆開來看：它怎麼算的、為什麼是 4%、在台灣適不適用。

## 🎯 學習目標
1. 了解 Trinity Study（1998）如何用歷史數據評估退休提領策略
2. 計算不同股債配置、提領率的30年生存機率
3. 理解「報酬順序風險」對退休規劃的致命影響

## 匯入套件

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'Heiti TC', 'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ 匯入完成")

## 一、什麼是 4% 法則？

> **每年提領金額 = 退休金總額 × 4%**

例：退休金 $2,500 萬台幣 → 每年可花 $100 萬（每月 $8.3 萬）

**背後邏輯：**
- 股市長期年化實質報酬約 5–7%
- 提領 4%，剩餘報酬讓資產繼續成長，不被通膨侵蝕

**最大陷阱 → 報酬序列風險（Sequence of Returns Risk）**
> 退休後「前幾年」的報酬比「後幾年」重要得多——因為你在從小池子裡持續提款

## 二、載入 Shiller 資料，計算月報酬

In [ ]:
import os

LOCAL = "data/shiller_data.csv"
if not os.path.exists(LOCAL):
    raise FileNotFoundError("請先執行 01_shiller_cape.ipynb 下載資料")

df = pd.read_csv(LOCAL, parse_dates=['date'], index_col='date')

# 實質月報酬 = 實質資本利得 + 股息率/12
df['div_yield_m'] = df['dividend'] / df['price'] / 12
df['stock_ret']   = df['real_price'].pct_change() + df['div_yield_m'].shift(1)
# 公債月報酬近似：年化殖利率 / 12
df['bond_ret']    = df['long_rate'].shift(1) / 100 / 12

df_sim = df.dropna(subset=['stock_ret', 'bond_ret']).copy()
print(f"可用資料：{df_sim.index[0].year}–{df_sim.index[-1].year}，共 {len(df_sim)} 個月")
print(f"股票實質月均報酬：{df_sim['stock_ret'].mean()*100:.3f}%")
print(f"公債月均報酬　　：{df_sim['bond_ret'].mean()*100:.3f}%")

## 三、模擬：各提領率在 30 年期間的存活率

In [ ]:
def simulate_retirement(returns_arr, wr, years=30):
    months = years * 12
    records = []
    for start in range(len(returns_arr) - months):
        p = 1.0
        monthly_w = wr / 12
        survived = True
        for m in range(months):
            p *= (1 + returns_arr[start + m])
            p -= monthly_w
            if p <= 0:
                survived = False
                break
        records.append({
            'start_year': df_sim.index[start].year,
            'survived': survived,
            'final': max(0.0, p)
        })
    return pd.DataFrame(records)

r_6040 = (df_sim['stock_ret'] * 0.6 + df_sim['bond_ret'] * 0.4).values
r_100s = df_sim['stock_ret'].values

withdrawal_rates = [0.03, 0.035, 0.04, 0.045, 0.05, 0.055, 0.06]
labels = ['3%', '3.5%', '4%', '4.5%', '5%', '5.5%', '6%']

surv_6040, surv_100s = [], []
for wr in withdrawal_rates:
    res = simulate_retirement(r_6040, wr)
    surv_6040.append(res['survived'].mean() * 100)
    res = simulate_retirement(r_100s, wr)
    surv_100s.append(res['survived'].mean() * 100)

print("30 年期存活率：")
print(f"{'提領率':>6} | {'60/40':>8} | {'100%股票':>8}")
print("-" * 32)
for lbl, s1, s2 in zip(labels, surv_6040, surv_100s):
    print(f"{lbl:>6} | {s1:>7.1f}% | {s2:>7.1f}%")

In [ ]:
x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, surv_6040, w, label='60/40 股債',  color='darkorange', alpha=0.85)
b2 = ax.bar(x + w/2, surv_100s, w, label='100% 股票', color='steelblue',  alpha=0.85)
ax.axhline(95, color='red', linestyle='--', linewidth=1.2, label='95% 安全門檻')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('30 年存活率（%）'); ax.set_ylim(40, 108)
ax.set_title('Trinity Study 重現：不同提領率的歷史存活率\n（Shiller 1881–今）', fontsize=12)
ax.legend()
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.0f}%',
            ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

## 四、哪些年份退休最危險？

存活率是「平均」，但個別退休年份差異極大。
最慘的年份是 **1960 年代末退休**：接下來是通膨飆升 + 股市停滯的「停滯性通膨」時代。

In [ ]:
res_6040 = simulate_retirement(r_6040, 0.04, years=30)

failed = res_6040[~res_6040['survived']]['start_year'].tolist()
print(f"4% 提領、60/40、30 年：失敗退休年份 → {failed if failed else '無，全部存活 ✅'}")

colors = ['#e74c3c' if not s else '#2ecc71' for s in res_6040['survived']]
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(res_6040['start_year'], res_6040['final'], color=colors, alpha=0.75, width=0.8)
ax.axhline(1, color='black', linestyle='--', linewidth=0.8, label='初始本金')
red_p   = mpatches.Patch(color='#e74c3c', alpha=0.75, label='30 年內耗盡（失敗）')
green_p = mpatches.Patch(color='#2ecc71', alpha=0.75, label='仍有剩餘（存活）')
ax.legend(handles=[green_p, red_p])
ax.set_xlabel('退休年份'); ax.set_ylabel('30 年後剩餘資產（初始=1）')
ax.set_title('4% 提領 × 60/40：各退休年份 30 年後剩餘資產', fontsize=12)
plt.tight_layout(); plt.show()

## 五、報酬序列風險 ★（最重要的概念）

兩個人，完全相同的 30 年月報酬——只是順序不同：
- 甲：一開始就遇熊市（報酬壞在前）
- 乙：最後才遇熊市（報酬壞在後）

**沒有提款時**：最終財富完全相同（乘法可交換）
**有提款時**：甲的資產可能提前耗盡！

In [ ]:
# 取 1966 年退休的真實歷史報酬（已知最艱難之一）
idx_1966 = int(np.where(df_sim.index.year == 1966)[0][0])
rets_bad = r_6040[idx_1966: idx_1966 + 360]
rets_rev = rets_bad[::-1]   # 相同報酬，但順序顛倒

def track(rets, wr=0.04):
    vals = [1.0]
    p = 1.0
    mw = wr / 12
    for r in rets:
        p = max(0.0, p * (1 + r) - mw)
        vals.append(p)
    return vals

def accum(rets):
    vals = [1.0]
    p = 1.0
    for r in rets:
        p *= (1 + r)
        vals.append(p)
    return vals

x = np.arange(361) / 12

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：純累積（無提款）—— 最終一樣
axes[0].plot(x, accum(rets_bad), label='壞報酬在前', color='#e74c3c')
axes[0].plot(x, accum(rets_rev), label='壞報酬在後', color='#2ecc71', linestyle='--')
axes[0].set_title('無提款：報酬順序不影響最終財富', fontsize=11)
axes[0].set_xlabel('持有年數'); axes[0].set_ylabel('資產倍數'); axes[0].legend()

# 右：每年提領 4% —— 差異巨大
tb = track(rets_bad); tr = track(rets_rev)
axes[1].plot(x, tb, label='壞報酬在前（退休初遇熊市）', color='#e74c3c')
axes[1].plot(x, tr, label='壞報酬在後（退休末遇熊市）', color='#2ecc71', linestyle='--')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('每年提領 4%：壞報酬在前 → 資產提前耗盡！', fontsize=11)
axes[1].set_xlabel('持有年數'); axes[1].set_ylabel('剩餘資產'); axes[1].legend(fontsize=8)

plt.suptitle('Sequence of Returns Risk（報酬序列風險）', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"壞報酬在前 → 30 年後剩餘：{tb[-1]:.2f}  {'✅ 存活' if tb[-1]>0 else '❌ 耗盡'}")
print(f"壞報酬在後 → 30 年後剩餘：{tr[-1]:.2f}  {'✅ 存活' if tr[-1]>0 else '❌ 耗盡'}")

## 六、討論

**結論：**
- 4% 提領在 145 年資料中，60/40 組合 30 年存活率 > 95%
- 最危險的退休年份集中在 1960 年代末（停滯性通膨時代）
- **報酬序列風險**：退休後頭幾年的報酬決定一切——同樣的長期報酬，順序不同就是生死之別

**思考問題：**
1. 如果提早退休（FIRE，例如 45 歲），需要撐 45 年，4% 法則還夠用嗎？
2. 遇到序列風險時，有哪些策略可以應對？（提示：動態提領、現金緩衝、part-time income）
3. 台灣退休族通常靠房租 + 儲蓄，這有序列風險嗎？跟股市型退休有何不同？
4. 通膨高時，「固定提領 4%」實際上是在不斷稀釋購買力。如何修正？

## 七、這跟你有什麼關係？

**用 4% 法則，反推你的退休目標數字。**

大多數人不知道自己要存多少錢——所以就一直沒有目標，一直拖延。
現在就算出來：退休後每月想花多少，就需要存多少。

> 🔧 把下面的數字換成你自己的情況，直接 Run

In [ ]:
# 🔧 改成你自己的數字
monthly_expense = 50_000   # 退休後每月想花多少（NT$）
current_age     = 25       # 現在幾歲
retire_age      = 60       # 預計退休年齡
annual_ret      = 0.07     # 年化報酬率（ETF 扣費後，保守估 5–7%）

# ── 計算退休目標 ──────────────────────────────────────────
annual_exp  = monthly_expense * 12
target_4pct = annual_exp / 0.04   # 4% 法則目標
target_3pct = annual_exp / 0.03   # 保守版（多存 33%）

years      = retire_age - current_age
months_inv = years * 12
mr         = (1 + annual_ret) ** (1/12) - 1

# 每月需存多少？PMT = FV × r / ((1+r)^n - 1)
pmt_4 = target_4pct * mr / ((1 + mr) ** months_inv - 1)
pmt_3 = target_3pct * mr / ((1 + mr) ** months_inv - 1)

print(f"退休後每月花費：NT${monthly_expense:,}  →  年花費 NT${annual_exp:,}")
print()
print(f"退休目標資產（4% 法則）：NT${target_4pct:>12,.0f}  （{target_4pct/10000:.0f} 萬）")
print(f"退休目標資產（3% 保守）：NT${target_3pct:>12,.0f}  （{target_3pct/10000:.0f} 萬）")
print()
print(f"從 {current_age} 歲投資到 {retire_age} 歲（{years} 年），年化 {annual_ret*100:.0f}%：")
print(f"  每月至少存（4% 目標）：NT${pmt_4:>8,.0f}")
print(f"  每月至少存（3% 保守）：NT${pmt_3:>8,.0f}")
print()
print("序列風險的應對建議：")
print(f"  {retire_age-5} 歲開始（退休前 5 年），把股票比例從 100% 逐年降到 60%")
print(f"  退休後的第一個熊市，是財務規劃最大的威脅——別在那時候賣股票")

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| 4% 法則 | 100% 股票配置，30年生存率歷史上超過 95% |
| 報酬順序風險 | 前幾年遇到熊市，即使整體報酬相同也可能破產 |
| 股票比例 | 股票比例越高，生存率越高（反直覺，但有據可查） |
| 個人應用 | 所需資產 = 年支出 × 25（4%法則的倒數） |

> **下一章：** 定期定額 vs 一次性投入——哪個策略更好？